# Jupyter Notebook for Visualizing Beam Parse Traces, Debugging, etc.

In [1]:
import os, sys, re
import string
import numpy as np
import torch
import subprocess

from utils import process_conll_data, tokenized_sent_to_Sentence, print_oracle_sequence, get_training_data
from parser import Vocabulary
from modeling import  train_model, get_data_loader, model, device, beam_parse, get_transition_and_vocab_logprobs, beam_step, logsumexp

In [2]:
pwd

'/mnt/DataDrive/ilcgdp/ilcgdp3/ilcgdp'

## Load up CoNLL sentences as Sentence objects, prepare a Vocabulary object, UNK-ify the Dev sentences

In [4]:
train_sentences = process_conll_data("../ptb_dependencies/universal_dependencies/train/ptb_train.conllu")

vocab = Vocabulary()
vocab.populate_vocabulary(train_sentences)

dev_sentences = process_conll_data("../ptb_dependencies/universal_dependencies/dev/ptb_dev.conllu")
vocab.unkify(dev_sentences)

## Sentence Objects

In [15]:
my_sent = dev_sentences[3]
my_sent.print_state()      #print Initial State parse state

Stack: []
Buffer: [(0, 'ROOT'), (1, 'officials'), (2, 'at'), (3, 'UNK'), (4, ','), (5, 'based'), (6, 'in'), (7, 'pittsburgh'), (8, ','), (9, 'declined'), (10, 'comment'), (11, '.')]
Dependencies: []
Is final configuration: False


In [21]:
#parse state features
my_sent.get_features_state(vocab)   #cf. vocab.idx2* below

array([   4,    1,    4,    1,    4,    1,    4,    1,    4,    1,    4,
          1,    4,    1,    4,    1,    4,    1,    4,    1,    4,    1,
          4,    1,    4,    1,    4,    1,    4,    1,    4,    1,    4,
          1,    4,    1,    4,    1,    4,    1,    4,    1,    4,    1,
          4,    1,    4,    1,    4,    1,    4,    1,    4,    1,    4,
          1,    4,    1,    4,    1,    4,    1,    4,    1,    4,    1,
          4,    1,    4,    1,    4,    1,    4,    1,    4,    1,    4,
          1,    4,    1,    4,    1,    0,    0,  145,    2,    4,    1,
       1013,    6,    4,    1,    4,    1,   17,   13,    4,    4],
      dtype=int32)

In [16]:
#one Oracle step
my_sent.get_oracle_transition()

'shift'

In [17]:
#complete Oracle sequence, showing all details
print_oracle_sequence(sentence = my_sent, train_vocab = vocab, num_transitions = -1, silence = False, print_stack_details = True)

Stack: []
Buffer: [(0, 'ROOT'), (1, 'officials'), (2, 'at'), (3, 'UNK'), (4, ','), (5, 'based'), (6, 'in'), (7, 'pittsburgh'), (8, ','), (9, 'declined'), (10, 'comment'), (11, '.')]
Dependencies: []
Is final configuration: False

shift

Stack: [[0]]
Buffer: [(1, 'officials'), (2, 'at'), (3, 'UNK'), (4, ','), (5, 'based'), (6, 'in'), (7, 'pittsburgh'), (8, ','), (9, 'declined'), (10, 'comment'), (11, '.')]
Dependencies: []
Is final configuration: False
Right Spine at stack index -1:
Token idx:0, Form:ROOT/0, and Dep:ROOT/0, Is Complete, Parent idx:-1, w/Left Children:


right_pred(root)

Stack: [[0, ('root', [])]]
Buffer: [(1, 'officials'), (2, 'at'), (3, 'UNK'), (4, ','), (5, 'based'), (6, 'in'), (7, 'pittsburgh'), (8, ','), (9, 'declined'), (10, 'comment'), (11, '.')]
Dependencies: []
Is final configuration: False
Right Spine at stack index -1:
Token idx:0, Form:ROOT/0, and Dep:ROOT/0, Parent idx:-1, w/Left Children:
Token idx:-1, Form:is_LDN/3, and Dep:root/6, Is Labeled Dummy, Paren

In [18]:
# `my_sent` has now been modified to the terminal state, in place
my_sent.print_state()

Stack: [[0, 9, 11]]
Buffer: []
Dependencies: [(3, 'punct', 4), (3, 'acl', 5), (7, 'case', 6), (5, 'obl', 7), (3, 'case', 2), (1, 'nmod', 3), (3, 'punct', 8), (9, 'obj', 10), (9, 'nsubj', 1), (0, 'root', 9), (9, 'punct', 11)]
Is final configuration: True


In [19]:
#print off CoNLL structure, based upon dependencies collected in the parse state
my_sent.print_conll()   #cf. my_sent.write_conll()

1	officials	_	_	_	_	9	nsubj	_	_

2	at	_	_	_	_	3	case	_	_

3	aristech	_	_	_	_	1	nmod	_	_

4	,	_	_	_	_	3	punct	_	_

5	based	_	_	_	_	3	acl	_	_

6	in	_	_	_	_	7	case	_	_

7	pittsburgh	_	_	_	_	5	obl	_	_

8	,	_	_	_	_	3	punct	_	_

9	declined	_	_	_	_	0	root	_	_

10	comment	_	_	_	_	9	obj	_	_

11	.	_	_	_	_	9	punct	_	_



## Vocabulary object attributes

In [6]:
vocab.form2idx    #cf. vocab.idx2form

{'ROOT': 0,
 'UNK': 1,
 'is_UDN': 2,
 'is_LDN': 3,
 'not_available': 4,
 'bush': 5,
 'and': 6,
 'gorbachev': 7,
 'will': 8,
 'hold': 9,
 'two': 10,
 'days': 11,
 'of': 12,
 'informal': 13,
 'talks': 14,
 'next': 15,
 'month': 16,
 '.': 17,
 'the': 18,
 'president': 19,
 'said': 20,
 'that': 21,
 'he': 22,
 'kremlin': 23,
 'leader': 24,
 'would': 25,
 'meet': 26,
 'dec.': 27,
 '2-3': 28,
 'aboard': 29,
 'u.s.': 30,
 'soviet': 31,
 'naval': 32,
 'vessels': 33,
 'in': 34,
 'mediterranean': 35,
 'to': 36,
 'discuss': 37,
 'a': 38,
 'wide': 39,
 'range': 40,
 'issues': 41,
 'without': 42,
 'formal': 43,
 'agenda': 44,
 'simultaneous': 45,
 'announcement': 46,
 'was': 47,
 'made': 48,
 'moscow': 49,
 'neither': 50,
 'nor': 51,
 'expected': 52,
 'any': 53,
 '``': 54,
 'substantial': 55,
 'decisions': 56,
 'or': 57,
 'agreements': 58,
 "''": 59,
 'seaborne': 60,
 'meetings': 61,
 'wo': 62,
 "n't": 63,
 'disrupt': 64,
 'plans': 65,
 'for': 66,
 'summit': 67,
 'spring': 68,
 'summer': 69,
 ',': 

In [7]:
vocab.dep2idx    #cf. vocab.idx2dep

{'ROOT': 0,
 'not_available': 1,
 'nsubj': 2,
 'cc': 3,
 'conj': 4,
 'aux': 5,
 'root': 6,
 'nummod': 7,
 'obj': 8,
 'case': 9,
 'amod': 10,
 'nmod': 11,
 'obl:tmod': 12,
 'punct': 13,
 'det': 14,
 'mark': 15,
 'compound': 16,
 'ccomp': 17,
 'obl': 18,
 'xcomp': 19,
 'nsubj:pass': 20,
 'aux:pass': 21,
 'cc:preconj': 22,
 'advmod': 23,
 'cop': 24,
 'acl:relcl': 25,
 'acl': 26,
 'fixed': 27,
 'nmod:poss': 28,
 'advcl': 29,
 'compound:prt': 30,
 'appos': 31,
 'obl:npmod': 32,
 'dep': 33,
 'iobj': 34,
 'det:predet': 35,
 'expl': 36,
 'orphan': 37,
 'parataxis': 38,
 'csubj': 39,
 'discourse': 40,
 'csubj:pass': 41}

In [8]:
vocab.transition2idx   #cf. vocab.idx2transition

{'shift': 0,
 'insert_as_head': 1,
 'insert_into_tree': 2,
 'left_pred(nsubj)': 3,
 'right_pred(nsubj)': 4,
 'left_comp(nsubj)': 5,
 'right_comp(nsubj)': 6,
 'left_pred(cc)': 7,
 'right_pred(cc)': 8,
 'left_comp(cc)': 9,
 'right_comp(cc)': 10,
 'left_pred(conj)': 11,
 'right_pred(conj)': 12,
 'left_comp(conj)': 13,
 'right_comp(conj)': 14,
 'left_pred(aux)': 15,
 'right_pred(aux)': 16,
 'left_comp(aux)': 17,
 'right_comp(aux)': 18,
 'left_pred(root)': 19,
 'right_pred(root)': 20,
 'left_comp(root)': 21,
 'right_comp(root)': 22,
 'left_pred(nummod)': 23,
 'right_pred(nummod)': 24,
 'left_comp(nummod)': 25,
 'right_comp(nummod)': 26,
 'left_pred(obj)': 27,
 'right_pred(obj)': 28,
 'left_comp(obj)': 29,
 'right_comp(obj)': 30,
 'left_pred(case)': 31,
 'right_pred(case)': 32,
 'left_comp(case)': 33,
 'right_comp(case)': 34,
 'left_pred(amod)': 35,
 'right_pred(amod)': 36,
 'left_comp(amod)': 37,
 'right_comp(amod)': 38,
 'left_pred(nmod)': 39,
 'right_pred(nmod)': 40,
 'left_comp(nmod)': 4

## PyTorch Model

In [22]:
train_sentences = process_conll_data("../ptb_dependencies/universal_dependencies/train/ptb_train.conllu")

vocab = Vocabulary()
vocab.populate_vocabulary(train_sentences)

In [23]:
BATCH_SIZE = 2048
FORM_EMBEDDING_SIZE = 300
DEP_EMBEDDING_SIZE = 50
NODE_SIZE = 500
SPINE_SIZE = 1000
HIDDEN_SIZE = 2000

MODEL_NAME = 35

In [24]:
my_model = model(num_form_embeddings = len(vocab.form2idx),
                 form_embedding_size = FORM_EMBEDDING_SIZE, 
                num_dep_embeddings=len(vocab.dep2idx), 
                 dep_embedding_size = DEP_EMBEDDING_SIZE,
                node_size = NODE_SIZE, 
                 spine_size = SPINE_SIZE,
                 hidden_size = HIDDEN_SIZE, 
                 num_transitions = len(vocab.transition2idx)).to(device)

my_model.load_state_dict(torch.load(f'../ptb_ud/models/{MODEL_NAME}.pth',weights_only=True))
my_model.eval()

model(
  (form_embeddings): Embedding(39552, 300)
  (dep_embeddings): Embedding(42, 50)
  (activation): ReLU()
  (node_former): Linear(in_features=1050, out_features=500, bias=True)
  (node_former_dropout): Dropout(p=0.2, inplace=False)
  (spine_former): Linear(in_features=2000, out_features=1000, bias=True)
  (spine_former_dropout): Dropout(p=0.2, inplace=False)
  (fc1): Linear(in_features=4300, out_features=2000, bias=True)
  (fc1_dropout): Dropout(p=0.2, inplace=False)
  (fc1_to_transitions): Linear(in_features=2000, out_features=163, bias=True)
  (fc1_to_vocabulary): Linear(in_features=2000, out_features=39552, bias=True)
)

## Calculate Labelled and Unlabelled Attachment Score for two CoNLL Files

In [25]:
gold_parses = '../UD_data/UD_English-GUM-master/en_gum-ud-dev.conllu'
system_parses = '../en_gum-ud/en_gum-ud-dev_beam_size2_model26_0_1575.conllx'

In [26]:
print(f"\nReading gold-standard parses at {gold_parses}")
#read in the gold standard sentences
f = open(gold_parses)
content = f.read()

gold_sentences = []
current_sentence = []
for line in content.split("\n"):
    if not line ==''  and not line[0] == '#':
        current_sentence.append(line)
        
    if line =='' and current_sentence:        
        gold_sentences.append(current_sentence)
        current_sentence = []
print(f"There are {len(gold_sentences)} gold-standard parses in this file.\n")



print(f"Reading the system parses at {system_parses}")
#read in the machine parse
f = open(system_parses)
content = f.read()

machine_sentences = []
current_sentence = []
for line in content.split("\n"):
    if not line =='':
        current_sentence.append(line)
    
    if line =='' and current_sentence:
        machine_sentences.append(current_sentence)
        current_sentence = []
print(f"There are {len(machine_sentences)} system parses in this file.\n")



print("Only scoring as many sentences as are in the system parse file.")
print("Dependencies with the 'punct' label are not being evaluated.\n")

#ONLY EVALUATE AS MUCH AS HAS BEEN PARSED FOR MACHINE SENTENCES
total = 0
unlabeled_correct = 0
labeled_correct = 0

for i in range(len(machine_sentences)):
    for j in range(len(machine_sentences[i])):
        
        #don't evaluate punctuation, a la Buys and Blunsom
        if gold_sentences[i][j].split("\t")[7] =='punct' or gold_sentences[i][j].split("\t")[1] in string.punctuation:
            continue
        
        if gold_sentences[i][j].split("\t")[6] == machine_sentences[i][j].split("\t")[6]:
            unlabeled_correct+=1
            if gold_sentences[i][j].split("\t")[7] == machine_sentences[i][j].split("\t")[7]:
                labeled_correct+=1
        #else:
        #    print(f"There was a disagreement in sentence: {i}, word: {j}")
        total +=1
        
print(f"Total number of dependencies: {total}\n")
print(f"Unlabeled Correct: {unlabeled_correct}")
print(f"Unlabeled Attachment: {round(unlabeled_correct/total,4)}\n")

print(f"Labeled Correct: {labeled_correct}")
print(f"Labeled Attachment: {round(labeled_correct/total,4)}")


Reading gold-standard parses at ../UD_data/UD_English-GUM-master/en_gum-ud-dev.conllu
There are 1575 gold-standard parses in this file.

Reading the system parses at ../en_gum-ud/en_gum-ud-dev_beam_size2_model26_0_1575.conllx
There are 1575 system parses in this file.

Only scoring as many sentences as are in the system parse file.
Dependencies with the 'punct' label are not being evaluated.

Total number of dependencies: 24795

Unlabeled Correct: 13286
Unlabeled Attachment: 0.5358

Labeled Correct: 11015
Labeled Attachment: 0.4442


In [27]:
gold_sentences

[['1\tIntroduction\tintroduction\tNOUN\tNN\tNumber=Sing\t0\troot\t0:root\tDiscourse=organization-heading:1->29:3:grf-ly-_-_|Entity=(1-abstract-new-nnnnn-cf1-1-sgl)|MSeg=Introduc-tion'],
 ['1\tResearch\tresearch\tNOUN\tNN\tNumber=Sing\t12\tnsubj\t12:nsubj\tDiscourse=context-background:2->14:2:sem-synym-4-8,108-109-_|Entity=(2-abstract-new-nnnns-cf2-1-coref',
  '2\ton\ton\tADP\tIN\t_\t7\tcase\t7:case\t_',
  '3\tadult\tadult\tNOUN\tNN\tNumber=Sing\t5\tcompound\t5:compound\tEntity=(3-abstract-new-sssss-cf1-5-coref|SpaceAfter=No|XML=<w>',
  '4\t-\t-\tPUNCT\tHYPH\t_\t3\tpunct\t3:punct\tSpaceAfter=No',
  '5\tlearned\tlearn\tVERB\tVBN\tTense=Past|VerbForm=Part|Voice=Pass\t7\tamod\t7:amod\tMSeg=learn-ed|XML=</w>',
  '6\tsecond\tsecond\tADJ\tJJ\tDegree=Pos|NumForm=Word|NumType=Ord\t7\tamod\t7:amod\t_',
  '7\tlanguage\tlanguage\tNOUN\tNN\tNumber=Sing\t1\tnmod\t1:nmod:on\tEntity=3)',
  '8\t(\t(\tPUNCT\t-LRB-\t_\t9\tpunct\t9:punct\tDiscourse=restatement-partial:3->2:0:sem-synym-4-8,10-_+grf-prn-9,1

In [28]:
machine_sentences

[['1\tintroduction\t_\t_\t_\t_\t0\troot\t_\t_'],
 ['1\tresearch\t_\t_\t_\t_\t12\tnsubj\t_\t_',
  '2\ton\t_\t_\t_\t_\t7\tcase\t_\t_',
  '3\tadult\t_\t_\t_\t_\t7\tcompound\t_\t_',
  '4\t-\t_\t_\t_\t_\t3\tpunct\t_\t_',
  '5\tlearned\t_\t_\t_\t_\t7\tcompound\t_\t_',
  '6\tsecond\t_\t_\t_\t_\t7\tamod\t_\t_',
  '7\tlanguage\t_\t_\t_\t_\t1\tnmod\t_\t_',
  '8\t(\t_\t_\t_\t_\t9\tpunct\t_\t_',
  '9\tl2\t_\t_\t_\t_\t7\tnmod:unmarked\t_\t_',
  '10\t)\t_\t_\t_\t_\t9\tpunct\t_\t_',
  '11\thas\t_\t_\t_\t_\t12\taux\t_\t_',
  '12\tprovided\t_\t_\t_\t_\t0\troot\t_\t_',
  '13\tconsiderable\t_\t_\t_\t_\t14\tamod\t_\t_',
  '14\tinsight\t_\t_\t_\t_\t12\tobj\t_\t_',
  '15\tinto\t_\t_\t_\t_\t19\tcase\t_\t_',
  '16\tthe\t_\t_\t_\t_\t19\tdet\t_\t_',
  '17\tneurocognitive\t_\t_\t_\t_\t19\tcompound\t_\t_',
  '18\tmechanisms\t_\t_\t_\t_\t19\tcompound\t_\t_',
  '19\tunderlying\t_\t_\t_\t_\t14\tnmod\t_\t_',
  '20\tthe\t_\t_\t_\t_\t21\tdet\t_\t_',
  '21\tlearning\t_\t_\t_\t_\t28\tnsubj\t_\t_',
  '22\tand\t_\t_\t_\t_\

## Beam Parsing

In [33]:
train_sentences = process_conll_data("../ptb_dependencies/universal_dependencies/train/ptb_train.conllu")

vocab = Vocabulary()
vocab.populate_vocabulary(train_sentences)

dev_sentences = process_conll_data("../ptb_dependencies/universal_dependencies/dev/ptb_dev.conllu")
vocab.unkify(dev_sentences)

In [34]:
BATCH_SIZE = 2048
FORM_EMBEDDING_SIZE = 300
DEP_EMBEDDING_SIZE = 50
NODE_SIZE = 500
SPINE_SIZE = 1000
HIDDEN_SIZE = 2000

MODEL_NAME = 35

In [35]:
my_model = model(num_form_embeddings = len(vocab.form2idx),
                 form_embedding_size = FORM_EMBEDDING_SIZE, 
                num_dep_embeddings=len(vocab.dep2idx), 
                 dep_embedding_size = DEP_EMBEDDING_SIZE,
                node_size = NODE_SIZE, 
                 spine_size = SPINE_SIZE,
                 hidden_size = HIDDEN_SIZE, 
                 num_transitions = len(vocab.transition2idx)).to(device)

my_model.load_state_dict(torch.load(f'../ptb_ud/models/{MODEL_NAME}.pth',weights_only=True))
my_model.eval()

model(
  (form_embeddings): Embedding(39552, 300)
  (dep_embeddings): Embedding(42, 50)
  (activation): ReLU()
  (node_former): Linear(in_features=1050, out_features=500, bias=True)
  (node_former_dropout): Dropout(p=0.2, inplace=False)
  (spine_former): Linear(in_features=2000, out_features=1000, bias=True)
  (spine_former_dropout): Dropout(p=0.2, inplace=False)
  (fc1): Linear(in_features=4300, out_features=2000, bias=True)
  (fc1_dropout): Dropout(p=0.2, inplace=False)
  (fc1_to_transitions): Linear(in_features=2000, out_features=163, bias=True)
  (fc1_to_vocabulary): Linear(in_features=2000, out_features=39552, bias=True)
)

### CoNLL sentence from the Dev Set

In [32]:
dev_sentences[3].print_state()

Stack: []
Buffer: [(0, 'ROOT'), (1, 'officials'), (2, 'at'), (3, 'UNK'), (4, ','), (5, 'based'), (6, 'in'), (7, 'pittsburgh'), (8, ','), (9, 'declined'), (10, 'comment'), (11, '.')]
Dependencies: []
Is final configuration: False


In [36]:
parsed = beam_parse(dev_sentences[3], my_model, vocab, beam_size=3, verbose=True, stat_file=None)

Num Transitions: 0
Stack: []
Buffer: [(0, 'ROOT'), (1, 'officials'), (2, 'at'), (3, 'UNK'), (4, ','), (5, 'based'), (6, 'in'), (7, 'pittsburgh'), (8, ','), (9, 'declined'), (10, 'comment'), (11, '.')]
Dependencies: []
Is final configuration: False
Parse log Probability: 0.0
-----------------------------------------------------


Num Transitions: 1
Log Prefix Probability from LogSumExp: -6.957269668579102
Normalized Log Probabilities: [np.float64(1.0)]
Stack: [[0]]
Buffer: [(1, 'officials'), (2, 'at'), (3, 'UNK'), (4, ','), (5, 'based'), (6, 'in'), (7, 'pittsburgh'), (8, ','), (9, 'declined'), (10, 'comment'), (11, '.')]
Dependencies: []
Is final configuration: False
Transition: shift, log Probability: -6.9573
-----------------------------------------------------


Num Transitions: 2
Log Prefix Probability from LogSumExp: -6.957289408737882
Normalized Log Probabilities: [np.float64(0.9999901767397782), np.float64(5.267188018049195e-06), np.float64(4.556072203768867e-06)]
Stack: [[0, ('r

/tmp/ipykernel_440341/2901323795.py:1: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:835.)
  parsed = beam_parse(dev_sentences[3], my_model, vocab, beam_size=3, verbose=True, stat_file=None)




Num Transitions: 12
Log Prefix Probability from LogSumExp: -32.586299194871266
Normalized Log Probabilities: [np.float64(0.7053825400017455), np.float64(0.24838087419656985), np.float64(0.04623658580168481)]
Stack: [[0, ('root', [])], [1, 3, 5, ('obl', [])]]
Buffer: [(6, 'in'), (7, 'pittsburgh'), (8, ','), (9, 'declined'), (10, 'comment'), (11, '.')]
Dependencies: [(3, 'punct', 4), (3, 'case', 2), (1, 'nmod', 3), (3, 'acl', 5)]
Is final configuration: False
Transition: right_comp(obl), log Probability: -32.9353
-----------------------------------------------------
Stack: [[0, ('root', [])], [1, ('nmod', [('case', 2)])], [3, 5, ('obl', [])]]
Buffer: [(6, 'in'), (7, 'pittsburgh'), (8, ','), (9, 'declined'), (10, 'comment'), (11, '.')]
Dependencies: [(3, 'punct', 4), (3, 'acl', 5)]
Is final configuration: False
Transition: right_comp(obl), log Probability: -33.9791
-----------------------------------------------------
Stack: [[0, ('root', [])], [1, 3, ('acl', [])], [5, ('obl', [])]]
Buf

In [37]:
parsed.print_state()

Stack: [[0, 9, 11]]
Buffer: []
Dependencies: [(3, 'punct', 4), (3, 'case', 2), (1, 'nmod', 3), (3, 'acl', 5), (7, 'case', 6), (5, 'obl', 7), (1, 'punct', 8), (9, 'obj', 10), (9, 'nsubj', 1), (0, 'root', 9), (9, 'punct', 11)]
Is final configuration: True


### Arbitrary Input Sentence

In [38]:
my_Sent = tokenized_sent_to_Sentence(['The','girl','kicked','the','ball','.'], vocab)
my_Sent.print_state()

Stack: []
Buffer: [(0, 'ROOT'), (1, 'the'), (2, 'girl'), (3, 'kicked'), (4, 'the'), (5, 'ball'), (6, '.')]
Dependencies: []
Is final configuration: False


In [39]:
parsed = beam_parse(my_Sent, my_model, vocab, beam_size=3, verbose=True, stat_file=None)

Num Transitions: 0
Stack: []
Buffer: [(0, 'ROOT'), (1, 'the'), (2, 'girl'), (3, 'kicked'), (4, 'the'), (5, 'ball'), (6, '.')]
Dependencies: []
Is final configuration: False
Parse log Probability: 0.0
-----------------------------------------------------


Num Transitions: 1
Log Prefix Probability from LogSumExp: -1.5164945125579834
Normalized Log Probabilities: [np.float64(1.0)]
Stack: [[0]]
Buffer: [(1, 'the'), (2, 'girl'), (3, 'kicked'), (4, 'the'), (5, 'ball'), (6, '.')]
Dependencies: []
Is final configuration: False
Transition: shift, log Probability: -1.5165
-----------------------------------------------------


Num Transitions: 2
Log Prefix Probability from LogSumExp: -1.5165204915860815
Normalized Log Probabilities: [np.float64(0.9999897401115437), np.float64(5.687364476545875e-06), np.float64(4.572523979639436e-06)]
Stack: [[0, ('root', [])]]
Buffer: [(1, 'the'), (2, 'girl'), (3, 'kicked'), (4, 'the'), (5, 'ball'), (6, '.')]
Dependencies: []
Is final configuration: False
Trans

In [40]:
parsed.print_state()

Stack: [[0, 3, 6]]
Buffer: []
Dependencies: [(2, 'det', 1), (5, 'det', 4), (3, 'obj', 5), (3, 'nsubj', 2), (0, 'root', 3), (3, 'punct', 6)]
Is final configuration: True
